# V6.5 External Expansion — Foreign Object + Broken Strand Staging / QA

Notebook này triển khai phần mở rộng dữ liệu V6.5 theo đúng hướng đã chốt.

**Foreign object**: lấy appearance từ Open Images V7 (`Balloon`, `Kite`, `Plastic bag`, `Bird`, `Person`), nhưng không merge generic full-image trực tiếp. Object được cắt ra và tạo **relation-aware synthetic candidates** trên power-line **TRAIN** background.

**Broken strand**: chỉ lấy nguồn conductor/power-line specific:
- MPCD / PowerLine-MTYOLO
- Power Equipment Image Dataset
  - `dx_dg` → `broken_strand`
  - `dx_sg` → review-only, không map thẳng
  - `yw` → `foreign_object`
  - `nw` → `bird_nest`

Notebook tạo **additive overlay + provenance + contact sheets + dedup QA**. Nó không tự promotion toàn bộ và không sửa TEST. Sanity train để OFF mặc định cho tới khi có exact canonical base YAML.

In [ ]:
# 0. CONFIG
from pathlib import Path
import os, sys, json, math, hashlib, shutil, subprocess, zipfile, tarfile
import random, re, xml.etree.ElementTree as ET
import numpy as np
import pandas as pd

W = Path('/kaggle/working')
ROOT = W / 'v6p5_external_expansion'
SRC = ROOT / 'sources'
BASE = ROOT / 'base_v6p2e'
OVERLAY = ROOT / 'v6p5_overlay'
QA = ROOT / 'qa'
CACHE = ROOT / 'cache'
for p in [ROOT,SRC,BASE,OVERLAY,QA,CACHE]: p.mkdir(parents=True, exist_ok=True)

NAMES = ['insulator','insulator_broken','insulator_flashover','bird_nest','foreign_object','broken_strand']

V6P2E_ROOT = 'stash://bunpmc/projects/yolov11sdi/datasets'
V6P2E_TAG = 'v6p2e_23562a45b343'

OI_CLASSES = ['Balloon','Kite','Plastic bag','Bird','Person']
OI_MAX_SAMPLES_PER_CLASS = 250
OI_SEED = 20261007
SYNTH_PER_SUBTYPE = 100
SYNTH_REQUIRE_MASK = True

POWER_EQUIP_HF_REPO = 'sxiong/Power-equipment-image-dataset'
POWER_EQUIP_FILENAME = 'transmission line object detection.zip'
POWER_EQUIP_TRAIN_FRAC = 0.80
MPCD_GDRIVE_ID = '1KyciMmwL2_p_-mwckHFqG5fkBA1jzATv'

POWER_MAP = {'dx_dg':5, 'yw':4, 'nw':3}
POWER_REVIEW_ONLY = {'dx_sg':'strand_loosening_review_only'}

RUN_PHASH = True
PHASH_MAX_HAMMING = 4
MAX_CONTACTS_PER_SOURCE = 80
MAX_SYNTH_CONTACTS = 100

UPLOAD_CAMBER = True
CAMBER_STAGE_ROOT = 'stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage'

RUN_SANITY_TRAIN = False
EXACT_BASE_DATA_YAML = None
SANITY_MODEL = 'yolo11n.pt'
SANITY_IMGSZ = 640
SANITY_EPOCHS = 10
SANITY_DEVICE = '0,1'

random.seed(OI_SEED); np.random.seed(OI_SEED)
print('ROOT:', ROOT)

In [ ]:
# 1. INSTALL + CAMBER AUTH
subprocess.check_call([sys.executable,'-m','pip','install','-q',
    'ultralytics==8.4.172','pyyaml','pillow','pandas','imagehash','tqdm',
    'gdown','huggingface_hub','fiftyone==1.22.1'])

import yaml, imagehash
from PIL import Image, ImageDraw, ImageFont
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
from ultralytics import YOLO
import fiftyone.utils.openimages as fouo

key = os.environ.get('CAMBER_API_KEY')
if not key:
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret('CAMBER_API_KEY')
    except Exception:
        key = None
if not key: raise RuntimeError('Bật Kaggle secret CAMBER_API_KEY trước.')
os.environ['CAMBER_API_KEY'] = key

camber_bin = Path.home()/'.camber/bin/camber'
if shutil.which('camber') is None and not camber_bin.exists():
    subprocess.check_call('curl -sL https://cli.cambercloud.com/install-v2.sh | bash', shell=True)
os.environ['PATH'] = f"{Path.home()}/.camber/bin:{Path.home()}/.local/bin:" + os.environ.get('PATH','')
CAMBER = shutil.which('camber') or str(camber_bin)
assert Path(CAMBER).exists(), 'Camber CLI not found'
print('CAMBER:', CAMBER)

In [ ]:
# 2. HELPERS
IMG_EXTS={'.jpg','.jpeg','.png','.bmp','.webp'}

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1024*1024), b''): h.update(b)
    return h.hexdigest()

def deterministic_fraction(key):
    h=hashlib.sha256(str(key).encode()).hexdigest()
    return int(h[:12],16)/float(16**12)

def stash_cp(remote, local, reuse=True):
    local=Path(local); local.parent.mkdir(parents=True, exist_ok=True)
    if reuse and local.exists() and local.stat().st_size>0: return local
    r=subprocess.run([CAMBER,'stash','cp',str(remote),str(local)],text=True,capture_output=True,env=os.environ.copy())
    if r.returncode!=0 or not local.exists():
        raise RuntimeError(f'Camber download failed: {remote}\n{r.stderr or r.stdout}')
    return local

def stash_put(local, remote):
    r=subprocess.run([CAMBER,'stash','cp',str(local),str(remote)],text=True,capture_output=True,env=os.environ.copy())
    if r.returncode!=0: raise RuntimeError(f'Camber upload failed: {remote}\n{r.stderr or r.stdout}')

def safe_copy(src,dst):
    dst=Path(dst); dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists(): shutil.copy2(src,dst)

def norm_names(names):
    if isinstance(names,dict): return [names[k] for k in sorted(names,key=lambda x:int(x))]
    return list(names or [])

def read_yolo_label(lp):
    out=[]
    if not Path(lp).exists(): return out
    for line in Path(lp).read_text(errors='ignore').splitlines():
        p=line.split()
        if len(p)<5: continue
        try: out.append((int(float(p[0])),*map(float,p[1:5])))
        except: pass
    return out

def write_yolo_label(lp, rows):
    Path(lp).parent.mkdir(parents=True, exist_ok=True)
    txt='\n'.join(f'{int(c)} {x:.8f} {y:.8f} {w:.8f} {h:.8f}' for c,x,y,w,h in rows if w>0 and h>0)
    Path(lp).write_text(txt+('\n' if txt else ''))

def yolo_xyxy(xc,yc,w,h): return [xc-w/2,yc-h/2,xc+w/2,yc+h/2]

def xyxy_to_yolo(b):
    x1,y1,x2,y2=map(float,b)
    x1=max(0,min(1,x1)); y1=max(0,min(1,y1)); x2=max(0,min(1,x2)); y2=max(0,min(1,y2))
    return [(x1+x2)/2,(y1+y2)/2,max(0,x2-x1),max(0,y2-y1)]

def make_contact(image_path, boxes, labels, out_path, title=''):
    im=Image.open(image_path).convert('RGB'); scale=min(1.0,1200/max(im.size))
    if scale<1: im=im.resize((int(im.width*scale),int(im.height*scale)))
    W,H=im.size; head=55
    canvas=Image.new('RGB',(W,H+head),'white'); canvas.paste(im,(0,head)); d=ImageDraw.Draw(canvas)
    try: font=ImageFont.truetype('DejaVuSans.ttf',16)
    except: font=ImageFont.load_default()
    d.text((8,8),title[:160],fill='black',font=font)
    for box,label in zip(boxes,labels):
        x1,y1,x2,y2=box; rect=[x1*W,y1*H+head,x2*W,y2*H+head]
        d.rectangle(rect,outline='red',width=4); d.text((rect[0]+2,max(head,rect[1]-18)),str(label),fill='red',font=font)
    Path(out_path).parent.mkdir(parents=True,exist_ok=True); canvas.save(out_path,quality=90)

In [ ]:
# 3. RECONSTRUCT IMMUTABLE V6.2E PARENT
PARENT_CACHE=BASE/'cache'; PARENT_RUNTIME=BASE/'runtime'; PARENT_CACHE.mkdir(parents=True,exist_ok=True)
freeze_path=PARENT_CACHE/'v6p2e.freeze.json'
stash_cp(f'{V6P2E_ROOT}/v6p2e/{V6P2E_TAG}/v6p2e.freeze.json',freeze_path)
parent_freeze=json.loads(freeze_path.read_text())
base_tar=PARENT_CACHE/f"base_{parent_freeze['base_tag']}.tar"
overlay_tar=PARENT_CACHE/f"overlay_{parent_freeze['overlay_tag']}.tar.gz"
recon_zip=PARENT_CACHE/'v6p2e_reconciled_labels.zip'; manifest_csv=PARENT_CACHE/'v6p2e_split_manifest.csv'
stash_cp(f"{V6P2E_ROOT}/curated_v6_rs1280_{parent_freeze['base_tag']}.tar",base_tar)
stash_cp(f"{V6P2E_ROOT}/v6p1/{parent_freeze['parent_dataset_tag']}/v6p1_fotl_overlay_{parent_freeze['overlay_tag']}.tar.gz",overlay_tar)
stash_cp(f'{V6P2E_ROOT}/v6p2e/{V6P2E_TAG}/v6p2e_reconciled_labels.zip',recon_zip)
stash_cp(f'{V6P2E_ROOT}/v6p2e/{V6P2E_TAG}/v6p2e_split_manifest.csv',manifest_csv)

BASE_EX=PARENT_CACHE/'base_extract'; OV_EX=PARENT_CACHE/'overlay_extract'; RECON=PARENT_CACHE/'reconciled'
for p in [BASE_EX,OV_EX,RECON]: p.mkdir(parents=True,exist_ok=True)
if not any(BASE_EX.rglob('data.yaml')):
    with tarfile.open(base_tar,'r:*') as t: t.extractall(BASE_EX)
if not any(OV_EX.rglob('data.yaml')):
    with tarfile.open(overlay_tar,'r:*') as t: t.extractall(OV_EX)
if not any(RECON.rglob('*.txt')):
    with zipfile.ZipFile(recon_zip) as z: z.extractall(RECON)

def find_sixclass_root(root):
    for yp in Path(root).rglob('data.yaml'):
        try:
            if norm_names(yaml.safe_load(yp.read_text()).get('names'))==NAMES: return yp.parent
        except: pass
    raise RuntimeError(f'No six-class root under {root}')

BASE_ROOT=find_sixclass_root(BASE_EX); OV_ROOT=find_sixclass_root(OV_EX); manifest=pd.read_csv(manifest_csv)

def locate_source(row):
    old_path=str(row['image_path_source']).replace('\\','/'); old_split=str(row['previous_v6p1_split']); marker=f'/images/{old_split}/'
    suffix=old_path.split(marker,1)[1]
    if suffix.startswith('base/'): root,rel=BASE_ROOT,suffix[len('base/'):]
    elif suffix.startswith('fotl/'): root,rel=OV_ROOT,suffix[len('fotl/'):]
    else: raise RuntimeError(suffix)
    return root/'images'/old_split/rel, root/'labels'/old_split/Path(rel).with_suffix('.txt')

def final_label(row,orig):
    rp=RECON/f"{row['sha256']}.txt"
    return rp if str(row.get('resolution','')).startswith('merged') and rp.exists() else orig

if not (PARENT_RUNTIME/'images/train').exists():
    for s in ['train','val','test']:
        (PARENT_RUNTIME/'images'/s).mkdir(parents=True,exist_ok=True); (PARENT_RUNTIME/'labels'/s).mkdir(parents=True,exist_ok=True)
    bad=[]
    for i,row in tqdm(manifest.iterrows(),total=len(manifest),desc='reconstruct V6.2E'):
        split=str(row['split']); ip,lp0=locate_source(row); lp=final_label(row,lp0)
        if not ip.exists() or not lp.exists(): bad.append((i,str(ip),str(lp))); continue
        name=f"{str(row['sha256'])[:16]}_{i:06d}{ip.suffix.lower()}"
        safe_copy(ip,PARENT_RUNTIME/'images'/split/name); safe_copy(lp,PARENT_RUNTIME/'labels'/split/Path(name).with_suffix('.txt'))
    if bad:
        pd.DataFrame(bad,columns=['row','image','label']).to_csv(QA/'parent_unresolved.csv',index=False)
        raise RuntimeError(f'Parent unresolved={len(bad)}')
print('V6.2E runtime ready')

In [ ]:
# 4. DOWNLOAD OPEN IMAGES V7 GENERIC FOREIGN OBJECT SOURCES (TRAIN ONLY)
OI_ROOT=SRC/'open_images_v7'; OI_ROOT.mkdir(parents=True,exist_ok=True)
avail=set(fouo.get_classes(version='v7')); seg_avail=set(fouo.get_segmentation_classes(version='v7'))
for cls in OI_CLASSES:
    assert cls in avail, f'Open Images class missing: {cls}'
    types=['detections']+(['segmentations'] if cls in seg_avail else [])
    print('OpenImages',cls,'seg=',cls in seg_avail)
    fouo.download_open_images_split(dataset_dir=str(OI_ROOT),split='train',version='v7',label_types=types,classes=[cls],shuffle=True,seed=OI_SEED,max_samples=OI_MAX_SAMPLES_PER_CLASS,num_workers=8)

In [ ]:
# 5. EXTRACT OPEN IMAGES OBJECT CROPS / MASKS
OI_CROPS=SRC/'oi_object_crops'; OI_CROPS.mkdir(parents=True,exist_ok=True); oi_rows=[]
for cls in OI_CLASSES:
    types=['detections']+(['segmentations'] if cls in seg_avail else [])
    imp=fouo.OpenImagesV7DatasetImporter(dataset_dir=str(OI_ROOT),label_types=types,classes=[cls],only_matching=True,shuffle=False)
    imp.setup(); count=0
    try:
        for sample in imp:
            image_path,metadata,labels=sample; image_path=Path(image_path)
            if not image_path.exists(): continue
            if not isinstance(labels,dict): labels={'detections':labels}
            cand=[]; segs=labels.get('segmentations')
            if segs is not None and hasattr(segs,'detections'):
                cand=[('segmentation',d) for d in segs.detections if d.label==cls]
            if not cand:
                dets=labels.get('detections')
                if dets is not None and hasattr(dets,'detections'):
                    cand=[('detection',d) for d in dets.detections if d.label==cls]
            if not cand: continue
            im=Image.open(image_path).convert('RGB'); W,H=im.size
            for j,(kind,det) in enumerate(cand):
                x,y,w,h=map(float,det.bounding_box); x1=max(0,int(x*W)); y1=max(0,int(y*H)); x2=min(W,int((x+w)*W)); y2=min(H,int((y+h)*H))
                if x2-x1<8 or y2-y1<8: continue
                crop=im.crop((x1,y1,x2,y2)); mask_arr=None
                if kind=='segmentation':
                    if getattr(det,'mask',None) is not None: mask_arr=np.asarray(det.mask)
                    elif getattr(det,'mask_path',None) and Path(det.mask_path).exists(): mask_arr=np.asarray(Image.open(det.mask_path).convert('L'))
                mask_ok=mask_arr is not None
                rgba=crop.convert('RGBA')
                if mask_ok:
                    mask=Image.fromarray((mask_arr>0).astype(np.uint8)*255).resize(crop.size,Image.Resampling.NEAREST); rgba.putalpha(mask)
                else: rgba.putalpha(Image.new('L',crop.size,255))
                subtype=cls.lower().replace(' ','_'); uid=hashlib.sha256(f'{image_path}|{cls}|{j}'.encode()).hexdigest()[:16]
                op=OI_CROPS/subtype/f'{uid}.png'; op.parent.mkdir(parents=True,exist_ok=True); rgba.save(op)
                oi_rows.append({'source':'open_images_v7','source_split':'train','source_image':str(image_path),'source_sha256':sha256_file(image_path),'foreign_subtype':subtype,'crop_path':str(op),'mask_available':mask_ok})
                count+=1
    finally: imp.close()
    print(cls,'crops=',count)
df_oi=pd.DataFrame(oi_rows); df_oi.to_csv(QA/'open_images_crop_manifest.csv',index=False)
display(df_oi.groupby(['foreign_subtype','mask_available']).size().unstack(fill_value=0))

In [ ]:
# 6. DOWNLOAD + PARSE MPCD
import gdown
MPCD_ZIP=SRC/'MPCD.zip'; MPCD_DIR=SRC/'MPCD'
if not MPCD_ZIP.exists():
    got=gdown.download(f'https://drive.google.com/uc?id={MPCD_GDRIVE_ID}',str(MPCD_ZIP),quiet=False)
    if got is None: raise RuntimeError(f'MPCD download failed; place file at {MPCD_ZIP}')
if not MPCD_DIR.exists():
    MPCD_DIR.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(MPCD_ZIP) as z: z.extractall(MPCD_DIR)

def resolve_yolo_yaml(yp):
    cfg=yaml.safe_load(Path(yp).read_text()); base=Path(cfg.get('path',Path(yp).parent)); base=base if base.is_absolute() else (Path(yp).parent/base).resolve(); return cfg,base,norm_names(cfg.get('names',[]))

def image_to_label(ip):
    s=str(ip).replace('\\','/'); return Path(s.replace('/images/','/labels/')).with_suffix('.txt') if '/images/' in s else None

mpcd_records=[]
for yp in MPCD_DIR.rglob('data.yaml'):
    try: cfg,base,src_names=resolve_yolo_yaml(yp)
    except: continue
    broken_ids=[i for i,n in enumerate(src_names) if any(k in str(n).lower().replace('_',' ').replace('-',' ') for k in ['broken','fracture','fractured','discontinu','strand break'])]
    if not broken_ids: continue
    print('MPCD yaml:',yp,'names=',src_names,'broken_ids=',broken_ids)
    for sk in ['train','val','test']:
        raw=cfg.get(sk)
        if raw is None: continue
        p=Path(raw); p=p if p.is_absolute() else base/p
        imgs=[x for x in p.rglob('*') if x.suffix.lower() in IMG_EXTS] if p.exists() and p.is_dir() else []
        for ip in imgs:
            lp=image_to_label(ip)
            if lp is None or not lp.exists(): continue
            mapped=[(5,xc,yc,w,h) for cid,xc,yc,w,h in read_yolo_label(lp) if cid in broken_ids]
            if mapped: mpcd_records.append({'source':'MPCD','source_split':sk,'source_image':str(ip),'source_label':str(lp),'mapped_rows':mapped,'license':'AGPL-3.0 project; verify upstream dataset terms'})
uniq={(r['source_split'],r['source_image']):r for r in mpcd_records}; mpcd_records=list(uniq.values())
print('MPCD mapped images=',len(mpcd_records))

In [ ]:
# 7. DOWNLOAD + PARSE POWER EQUIPMENT OBJECT DETECTION
from huggingface_hub import hf_hub_download
PE_ZIP=SRC/'power_equipment_transmission_line_object_detection.zip'; PE_DIR=SRC/'power_equipment_object_detection'
if not PE_ZIP.exists():
    hp=hf_hub_download(repo_id=POWER_EQUIP_HF_REPO,filename=POWER_EQUIP_FILENAME,repo_type='dataset',local_dir=str(SRC/'hf_power_equipment'))
    shutil.copy2(hp,PE_ZIP)
if not PE_DIR.exists():
    PE_DIR.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(PE_ZIP) as z: z.extractall(PE_DIR)

imgs_by_stem={}
for ip in PE_DIR.rglob('*'):
    if ip.suffix.lower() in IMG_EXTS: imgs_by_stem.setdefault(ip.stem,[]).append(ip)
pe_records=[]; pe_review=[]
for xp in tqdm(list(PE_DIR.rglob('*.xml')),desc='PowerEquipment XML'):
    try: root=ET.parse(xp).getroot()
    except: continue
    fn=root.findtext('filename'); cand=(xp.parent/fn) if fn else None
    if cand is None or not cand.exists():
        hits=imgs_by_stem.get(Path(fn).stem if fn else xp.stem,[]); cand=hits[0] if hits else None
    if cand is None or not Path(cand).exists(): continue
    size=root.find('size')
    if size is not None: W=int(float(size.findtext('width'))); H=int(float(size.findtext('height')))
    else:
        with Image.open(cand) as im: W,H=im.size
    mapped=[]
    for obj in root.findall('object'):
        name=(obj.findtext('name') or '').strip(); bb=obj.find('bndbox')
        if bb is None: continue
        xmin,ymin,xmax,ymax=[float(bb.findtext(k)) for k in ['xmin','ymin','xmax','ymax']]
        box=[xmin/W,ymin/H,xmax/W,ymax/H]
        if name in POWER_MAP:
            xc,yc,w,h=xyxy_to_yolo(box); mapped.append((POWER_MAP[name],xc,yc,w,h))
        if name in POWER_REVIEW_ONLY:
            pe_review.append({'source_image':str(cand),'source_xml':str(xp),'source_class':name,'review_role':POWER_REVIEW_ONLY[name],'bbox_xyxy':box})
    if mapped:
        split='train' if deterministic_fraction(str(cand))<POWER_EQUIP_TRAIN_FRAC else 'external_eval'
        pe_records.append({'source':'PowerEquipment','source_split':split,'source_image':str(cand),'source_label':str(xp),'mapped_rows':mapped,'license':'MIT repository; verify original data redistribution terms'})
pd.DataFrame(pe_review).to_csv(QA/'power_equipment_review_only_dx_sg.csv',index=False)
print('PowerEquipment mapped images=',len(pe_records),'dx_sg review=',len(pe_review))

In [ ]:
# 8. BUILD DIRECT POWER-LINE OVERLAY
for s in ['train','external_eval']:
    (OVERLAY/'direct/images'/s).mkdir(parents=True,exist_ok=True); (OVERLAY/'direct/labels'/s).mkdir(parents=True,exist_ok=True)
direct=[]

def add_direct(rec,target):
    ip=Path(rec['source_image']); sha=sha256_file(ip); name=f"{rec['source'].lower()}_{sha[:16]}{ip.suffix.lower()}"
    di=OVERLAY/'direct/images'/target/name; dl=OVERLAY/'direct/labels'/target/Path(name).with_suffix('.txt')
    safe_copy(ip,di); write_yolo_label(dl,rec['mapped_rows']); classes=sorted({int(x[0]) for x in rec['mapped_rows']})
    direct.append({'overlay_image':str(di),'overlay_label':str(dl),'target_split':target,'source':rec['source'],'source_split':rec['source_split'],'source_image':str(ip),'source_sha256':sha,'mapped_classes':','.join(map(str,classes)),'mapped_class_names':','.join(NAMES[c] for c in classes),'n_mapped_boxes':len(rec['mapped_rows']),'license_note':rec.get('license','')})

for r in mpcd_records: add_direct(r,'train' if r['source_split']=='train' else 'external_eval')
for r in pe_records: add_direct(r,r['source_split'])
df_direct=pd.DataFrame(direct); df_direct.to_csv(QA/'direct_overlay_manifest.csv',index=False)
display(df_direct.groupby(['source','target_split','mapped_class_names']).size().reset_index(name='images'))

In [ ]:
# 9. INTERNAL TRAIN STRUCTURAL ANCHORS
anchors=[]; ti=PARENT_RUNTIME/'images/train'; tl=PARENT_RUNTIME/'labels/train'
for ip in tqdm(list(ti.glob('*')),desc='find insulator anchors'):
    if ip.suffix.lower() not in IMG_EXTS: continue
    lp=tl/ip.with_suffix('.txt').name
    for j,(cid,xc,yc,w,h) in enumerate(read_yolo_label(lp)):
        if cid==0: anchors.append({'image':str(ip),'label':str(lp),'anchor_xc':xc,'anchor_yc':yc,'anchor_w':w,'anchor_h':h})
df_anchor=pd.DataFrame(anchors)
print('anchors=',len(df_anchor),'unique backgrounds=',df_anchor['image'].nunique())

In [ ]:
# 10. RELATION-AWARE SYNTHETIC FOREIGN OBJECTS
SYN=OVERLAY/'synthetic_foreign'; (SYN/'images/train').mkdir(parents=True,exist_ok=True); (SYN/'labels/train').mkdir(parents=True,exist_ok=True)

def resize_rgba(obj,longside):
    ow,oh=obj.size; sc=longside/max(ow,oh); return obj.resize((max(2,int(ow*sc)),max(2,int(oh*sc))),Image.Resampling.LANCZOS)

def compose(bg,obj,a,subtype,rng):
    W,H=bg.size; ax,ay,aw,ah=a; acx,acy=ax*W,ay*H; along=max(aw*W,ah*H)
    if subtype=='person': rel=rng.uniform(.45,.90)
    elif subtype in {'balloon','kite'}: rel=rng.uniform(.20,.55)
    elif subtype=='bird': rel=rng.uniform(.10,.28)
    else: rel=rng.uniform(.12,.35)
    obj=resize_rgba(obj,max(12,min(int(along*rel),int(min(W,H)*.30))))
    if subtype=='person': dx=rng.uniform(-.8,.8)*max(aw*W,40); dy=rng.uniform(.35,1.2)*max(ah*H,40)
    else: dx=rng.uniform(-.7,.7)*max(aw*W,35); dy=rng.uniform(-.6,.6)*max(ah*H,35)
    x1=int(acx+dx-obj.width/2); y1=int(acy+dy-obj.height/2); x1=max(0,min(W-obj.width,x1)); y1=max(0,min(H-obj.height,y1)); x2=x1+obj.width; y2=y1+obj.height
    if x2<=x1 or y2<=y1: return None
    out=bg.convert('RGBA'); out.alpha_composite(obj,(x1,y1)); return out.convert('RGB'),[x1/W,y1/H,x2/W,y2/H]

usable=df_oi[df_oi['mask_available']==True].copy() if SYNTH_REQUIRE_MASK else df_oi.copy(); rng=random.Random(OI_SEED); syn=[]
for subtype in sorted(usable['foreign_subtype'].unique()):
    pool=usable[usable['foreign_subtype']==subtype].to_dict('records'); made=0; attempts=0
    while made<SYNTH_PER_SUBTYPE and attempts<SYNTH_PER_SUBTYPE*20:
        attempts+=1; objrec=rng.choice(pool); a=df_anchor.sample(1,random_state=rng.randint(0,10**9)).iloc[0]
        try: bg=Image.open(a['image']).convert('RGB'); obj=Image.open(objrec['crop_path']).convert('RGBA')
        except: continue
        res=compose(bg,obj,(float(a['anchor_xc']),float(a['anchor_yc']),float(a['anchor_w']),float(a['anchor_h'])),subtype,rng)
        if res is None: continue
        sim,box=res; rows=read_yolo_label(a['label']); fx,fy,fw,fh=xyxy_to_yolo(box); rows.append((4,fx,fy,fw,fh))
        uid=hashlib.sha256(f"{a['image']}|{objrec['crop_path']}|{made}|{OI_SEED}".encode()).hexdigest()[:18]
        di=SYN/'images/train'/f'syn_{subtype}_{uid}.jpg'; dl=SYN/'labels/train'/f'syn_{subtype}_{uid}.txt'; sim.save(di,quality=92); write_yolo_label(dl,rows)
        syn.append({'overlay_image':str(di),'overlay_label':str(dl),'source':'OpenImagesV7_synthetic_relation','foreign_subtype':subtype,'oi_source_image':objrec['source_image'],'oi_source_sha256':objrec['source_sha256'],'oi_crop':objrec['crop_path'],'background_image':a['image'],'background_sha256':sha256_file(a['image']),'anchor_class':'insulator','foreign_box_xyxy':box,'synthetic':True,'train_only':True}); made+=1
    print(subtype,'made=',made)
df_synth=pd.DataFrame(syn); df_synth.to_csv(QA/'synthetic_foreign_manifest.csv',index=False); display(df_synth['foreign_subtype'].value_counts())

In [ ]:
# 11. CONTACT SHEETS
CONTACT=QA/'contacts'; CONTACT.mkdir(parents=True,exist_ok=True)
for source in df_direct['source'].unique():
    for _,r in df_direct[df_direct['source']==source].head(MAX_CONTACTS_PER_SOURCE).iterrows():
        boxes=[]; labs=[]
        for cid,xc,yc,w,h in read_yolo_label(r['overlay_label']): boxes.append(yolo_xyxy(xc,yc,w,h)); labs.append(NAMES[cid])
        make_contact(r['overlay_image'],boxes,labs,CONTACT/f"{source}_{Path(r['overlay_image']).stem}.jpg",f"{source} | {r['target_split']} | {r['mapped_class_names']}")
for _,r in df_synth.head(MAX_SYNTH_CONTACTS).iterrows():
    boxes=[]; labs=[]
    for cid,xc,yc,w,h in read_yolo_label(r['overlay_label']):
        if cid==4: boxes.append(yolo_xyxy(xc,yc,w,h)); labs.append('foreign_object:'+r['foreign_subtype'])
    make_contact(r['overlay_image'],boxes,labs,CONTACT/f"synthetic_{Path(r['overlay_image']).stem}.jpg",f"synthetic relation | {r['foreign_subtype']}")
print('contacts:',CONTACT)

In [ ]:
# 12. DEDUP QA: direct external vs internal V6.2E
base_hash={}
for split in ['train','val','test']:
    for ip in tqdm(list((PARENT_RUNTIME/'images'/split).glob('*')),desc=f'base SHA {split}'):
        if ip.suffix.lower() in IMG_EXTS: base_hash.setdefault(sha256_file(ip),[]).append((split,str(ip)))
exact=[]
for ip in tqdm([Path(x) for x in df_direct['overlay_image'].tolist()],desc='direct exact SHA'):
    s=sha256_file(ip)
    for split,bp in base_hash.get(s,[]): exact.append({'external_image':str(ip),'base_split':split,'base_image':bp,'sha256':s})
df_exact=pd.DataFrame(exact); df_exact.to_csv(QA/'external_vs_internal_exact_duplicates.csv',index=False); print('exact=',len(df_exact))

near=[]
if RUN_PHASH:
    phbase=[]
    for split in ['train','val','test']:
        for ip in tqdm(list((PARENT_RUNTIME/'images'/split).glob('*')),desc=f'base pHash {split}'):
            if ip.suffix.lower() not in IMG_EXTS: continue
            try:
                with Image.open(ip) as im: hv=int(str(imagehash.phash(im.convert('RGB'),hash_size=8)),16)
                phbase.append((split,str(ip),hv))
            except: pass
    widths=[13,13,13,13,12]; shifts=[]; cur=0
    for w in widths: shifts.append(cur); cur+=w
    buckets={}
    for idx,(_,_,hv) in enumerate(phbase):
        for band,(w,sh) in enumerate(zip(widths,shifts)): buckets.setdefault((band,(hv>>sh)&((1<<w)-1)),[]).append(idx)
    for ip in tqdm([Path(x) for x in df_direct['overlay_image'].tolist()],desc='external pHash'):
        try:
            with Image.open(ip) as im: hv=int(str(imagehash.phash(im.convert('RGB'),hash_size=8)),16)
        except: continue
        cand=set()
        for band,(w,sh) in enumerate(zip(widths,shifts)): cand.update(buckets.get((band,(hv>>sh)&((1<<w)-1)),[]))
        for idx in cand:
            split,bp,bh=phbase[idx]; hd=(hv^bh).bit_count()
            if hd<=PHASH_MAX_HAMMING: near.append({'external_image':str(ip),'base_split':split,'base_image':bp,'phash_hamming':hd})
df_near=pd.DataFrame(near); df_near.to_csv(QA/'external_vs_internal_phash_candidates.csv',index=False); print('near=',len(df_near))

In [ ]:
# 13. REVIEW MANIFEST
review=[]
for _,r in df_direct.iterrows():
    review.append({'candidate_type':'direct_powerline','overlay_image':r['overlay_image'],'overlay_label':r['overlay_label'],'source':r['source'],'target_split':r['target_split'],'mapped_class_names':r['mapped_class_names'],'subtype':'','recommended_action':'REVIEW_THEN_KEEP' if r['target_split']=='train' else 'KEEP_AS_EXTERNAL_EVAL','decision':'','reviewer':'','comment':''})
for _,r in df_synth.iterrows():
    review.append({'candidate_type':'synthetic_relation','overlay_image':r['overlay_image'],'overlay_label':r['overlay_label'],'source':r['source'],'target_split':'train','mapped_class_names':'foreign_object','subtype':r['foreign_subtype'],'recommended_action':'VISUAL_REVIEW_THEN_KEEP','decision':'','reviewer':'','comment':''})
df_review=pd.DataFrame(review); df_review.insert(0,'review_id',[f'V65R{i:06d}' for i in range(len(df_review))]); df_review.to_csv(QA/'v6p5_candidate_review_manifest.csv',index=False)
display(df_review.groupby(['candidate_type','source','target_split','mapped_class_names']).size().reset_index(name='n'))

In [ ]:
# 14. FREEZE + PACKAGE + CAMBER

def dir_digest(root):
    items=[]
    for p in sorted(Path(root).rglob('*')):
        if p.is_file(): items.append(f'{p.relative_to(root)}:{sha256_file(p)}')
    return hashlib.sha256('\n'.join(items).encode()).hexdigest()

digest=dir_digest(OVERLAY); tag=f'v6p5_ext_{digest[:12]}'
freeze={'tag':tag,'taxonomy':{i:n for i,n in enumerate(NAMES)},'base_reference':{'dataset':'V6.2E','tag':V6P2E_TAG,'role':'dedup + internal TRAIN backgrounds only'},'sources':{'OpenImagesV7':{'classes':OI_CLASSES,'split':'train','role':'appearance source for relation-aware synthetic foreign objects'},'MPCD':{'role':'broken_strand direct power-line source'},'PowerEquipment':{'map':POWER_MAP,'review_only':POWER_REVIEW_ONLY}},'counts':{'direct_train_images':int((df_direct['target_split']=='train').sum()),'direct_external_eval_images':int((df_direct['target_split']=='external_eval').sum()),'synthetic_train_images':int(len(df_synth)),'review_candidates':int(len(df_review)),'exact_external_internal_duplicates':int(len(df_exact)),'phash_candidates':int(len(df_near))},'policy':{'generic_full_images_direct_merge':False,'synthetic_background_split':'internal train only','dx_sg_direct_merge':False,'test_modified':False,'auto_promotion':False},'overlay_sha256':digest}
freeze_path=ROOT/f'{tag}.freeze.json'; freeze_path.write_text(json.dumps(freeze,indent=2)); print(json.dumps(freeze,indent=2))

overlay_tar=ROOT/f'{tag}_overlay.tar.gz'; qa_zip=ROOT/f'{tag}_qa.zip'
if not overlay_tar.exists():
    with tarfile.open(overlay_tar,'w:gz') as t: t.add(OVERLAY,arcname='v6p5_overlay')
if not qa_zip.exists():
    with zipfile.ZipFile(qa_zip,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for p in QA.rglob('*'):
            if p.is_file(): z.write(p,arcname=str(p.relative_to(QA)))
print('TAG:',tag,'overlay MB=',overlay_tar.stat().st_size/1024**2,'qa MB=',qa_zip.stat().st_size/1024**2)
if UPLOAD_CAMBER:
    rr=f'{CAMBER_STAGE_ROOT}/{tag}'
    for p in [overlay_tar,qa_zip,freeze_path]: stash_put(p,f'{rr}/{p.name}')
    print('Camber root:',rr)

In [ ]:
# 15. OPTIONAL SANITY TRAIN — OFF BY DEFAULT
if RUN_SANITY_TRAIN:
    if not EXACT_BASE_DATA_YAML: raise RuntimeError('Set EXACT_BASE_DATA_YAML to exact canonical base YAML first.')
    base_yaml=Path(EXACT_BASE_DATA_YAML); assert base_yaml.exists(),base_yaml
    rv=pd.read_csv(QA/'v6p5_candidate_review_manifest.csv'); approved=rv[(rv['target_split']=='train') & (rv['decision'].astype(str).str.upper()=='KEEP')]
    if len(approved)==0: raise RuntimeError('Review manifest first and mark accepted rows decision=KEEP.')
    cfg=yaml.safe_load(base_yaml.read_text()); base_root=Path(cfg.get('path',base_yaml.parent)); base_root=base_root if base_root.is_absolute() else (base_yaml.parent/base_root).resolve()
    view=ROOT/'sanity_train_view'
    if view.exists(): shutil.rmtree(view)
    for s in ['train','val']:
        (view/'images'/s).mkdir(parents=True,exist_ok=True); (view/'labels'/s).mkdir(parents=True,exist_ok=True)
        src_i=Path(cfg[s]); src_i=src_i if src_i.is_absolute() else base_root/src_i; src_l=Path(str(src_i).replace('/images/','/labels/'))
        for ip in tqdm([p for p in src_i.rglob('*') if p.suffix.lower() in IMG_EXTS],desc=f'copy {s}'):
            rel=ip.relative_to(src_i); safe_copy(ip,view/'images'/s/rel); lp=src_l/rel.with_suffix('.txt');
            if lp.exists(): safe_copy(lp,view/'labels'/s/rel.with_suffix('.txt'))
    for _,r in approved.iterrows():
        ip=Path(r['overlay_image']); lp=Path(r['overlay_label']); name=f"v65_{hashlib.sha256(str(ip).encode()).hexdigest()[:12]}_{ip.name}"
        safe_copy(ip,view/'images/train'/name); safe_copy(lp,view/'labels/train'/Path(name).with_suffix('.txt'))
    ty=view/'data.yaml'; ty.write_text(yaml.safe_dump({'path':str(view),'train':'images/train','val':'images/val','names':{i:n for i,n in enumerate(NAMES)}},sort_keys=False))
    YOLO(SANITY_MODEL).train(data=str(ty),imgsz=SANITY_IMGSZ,epochs=SANITY_EPOCHS,batch=-1,device=SANITY_DEVICE,workers=8,project=str(ROOT/'sanity_runs'),name=tag,exist_ok=True,pretrained=True,seed=OI_SEED)
else:
    print('RUN_SANITY_TRAIN=False. Review overlay first; this is intentional.')

## Sau khi chạy xong

Ưu tiên xem:
- `qa/direct_overlay_manifest.csv`
- `qa/open_images_crop_manifest.csv`
- `qa/synthetic_foreign_manifest.csv`
- `qa/power_equipment_review_only_dx_sg.csv`
- `qa/external_vs_internal_exact_duplicates.csv`
- `qa/external_vs_internal_phash_candidates.csv`
- `qa/v6p5_candidate_review_manifest.csv`
- `qa/contacts/`
- `<tag>.freeze.json`

Không promotion toàn bộ tự động. `dx_sg` không được map sang `broken_strand`. Open Images synthetic phải nhìn contact sheet rồi mới `KEEP`.